# Qwen-Image-2.1 + psoriasis LoRA: one sample

Loads the text-to-image LoRA used in the Figure 3 sweep (checkpoint 1500) and generates one image.
Models load from `HF_HOME` (set it to the RAM copy, e.g. `/dev/shm/hf`, before starting Jupyter); the LoRA is read from `/workspace`.

In [ ]:
import os, torch
from diffusers import QwenImage21Pipeline

GPU = 0
LABEL = "psoriasis"
LORA = f"/workspace/synderm/qwen_lora/t2i/{LABEL}/r16-lr0.0001-512px-2000steps/checkpoint-1500/pytorch_lora_weights.safetensors"
PROMPT = f"An image of {LABEL}, a skin disease"   # the paper's prompt, as in the sweep
SEED = 0
print("HF_HOME =", os.environ.get("HF_HOME"), "| LoRA exists:", os.path.exists(LORA))

In [ ]:
pipe = QwenImage21Pipeline.from_pretrained("Qwen/Qwen-Image-2.1", dtype=torch.bfloat16).to(f"cuda:{GPU}")
pipe.load_lora_weights(os.path.dirname(LORA), weight_name=os.path.basename(LORA))
pipe.set_progress_bar_config(disable=True)

In [ ]:
image = pipe(prompt=PROMPT, height=512, width=512, num_inference_steps=40,
             generator=torch.Generator("cpu").manual_seed(SEED)).images[0]
image

In [ ]:
image.save(f"/workspace/synderm/qwen_lora/sample_{LABEL}_ckpt1500_seed{SEED}.png")